# Substructure Type Gallery — one of each, straight to a Rhino file

**One 3-span bridge carrying every substructure type civilpy places: a
semi-integral abutment, a hammerhead pier, a capped-pile bent, and an
integral abutment — designed, emitted, and baked into a real `.3dm`
without opening Rhino.**

During prototyping these models were drawn into a live Rhino 8 session
over the **Rhino MCP server**: the same tagged BrIM emit this notebook
builds, serialized with `emit_to_json` and executed by the draw script
inside the open document. The geometry pipeline was never MCP-specific —
MCP was only the transport.

On a network where no MCP server is available, the same models come from
the two offline backends, which need nothing but Python:

1. **`emit_to_3dm`** — bakes the emit into a real `.3dm` with standalone
   [`rhino3dm`](https://pypi.org/project/rhino3dm/) (`pip install rhino3dm`).
   No Rhino session, no server, no network. Open the file in Rhino and the
   full model is there — layers, colors, solids, rebar, and every `bim.*` /
   `pay.*` / `mat.*` user-text tag.
2. **`emit_to_json` + `Rhino Components/draw_bim_emit.py`** — the
   live-document twin: run the script in Rhino 8's ScriptEditor with
   `EMIT_JSON_PATH` pointed at the payload and the model is drawn into the
   open document (lofted breps, viewport setup, document string table).

> Units: feet, kips. Frame: X = stations, Y = transverse (girder 1 at
> y = 0), Z = 0 at top of deck at the crown.

In [1]:
from civilpy.structural.bridge_layout import (
    BridgeInput, layout_bridge, girder_line_loads)
from civilpy.structural.continuous_beam import ContinuousBeam
from civilpy.structural.substructure import substructure_units

inp = BridgeInput(spans_ft=(60.0, 80.0, 60.0), girder_count=4,
                  girder_spacing_ft=9.0, girder_label="W36X150",
                  overhang_ft=2.5, railing="SBR-1-20", cross_slope_pct=2.0)
layout = layout_bridge(inp)
units = substructure_units(layout)

print(f"{layout.total_length_ft:.0f} ft crossing, deck {layout.deck_width_ft:.1f} ft wide")
for u in units:
    print(f"  {u.name:<12} at station {u.station_ft:5.0f} ft")

200 ft crossing, deck 32.0 ft wide
  Abutment 1   at station     0 ft
  Pier 2       at station    60 ft
  Pier 3       at station   140 ft
  Abutment 2   at station   200 ft


## 1. Strength I reactions per support line — native, no MIDAS

Four support lines get four different substructure types, so each needs its
factored design reaction. `ContinuousBeam` gives the exact dead-load
reactions per girder line (DC1/DC2/DW factored separately), and the HL-93
lane load rides the same influence machinery. The unit totals then spread
onto the bearings by tributary width — the same pattern the *Substructure
Design from Preliminary Reactions* notebook uses with MIDAS reactions, so
swapping these numbers for a solved-model report changes nothing
downstream.

In [2]:
stz = [0.0]
for s in inp.spans_ft:
    stz.append(stz[-1] + s)

R_u = [0.0] * len(stz)
for g in range(inp.girder_count):
    w = girder_line_loads(layout, g)
    for case, f in (("dc1", 1.25), ("dc2", 1.25), ("dw", 1.5)):
        beam = ContinuousBeam(stz)
        beam.add_udl(w[case])
        R_u = [r + f * x for r, x in zip(R_u, beam.reactions())]

n_lanes = 2                       # ~29 ft roadway between SBR-1-20 faces
beam = ContinuousBeam(stz)
beam.add_udl(-0.64 * n_lanes)     # HL-93 lane load, both lanes
R_u = [abs(r) + 1.75 * abs(x) for r, x in zip(R_u, beam.reactions())]

# bearing loads by tributary width across the four girder lines
s_g, oh = inp.girder_spacing_ft, inp.overhang_ft
trib = [oh + s_g/2] + [s_g] * (inp.girder_count - 2) + [oh + s_g/2]
shares = [t / sum(trib) for t in trib]
bearing_y = [g * s_g for g in range(inp.girder_count)]

def bearing_loads(i):
    return [float(round(R_u[i] * f, 1)) for f in shares]

print("Strength I unit reactions (kip):",
      {u.name: round(R_u[u.index]) for u in units})
print("Pier bearing loads (kip):    ", bearing_loads(1))
print("Abutment bearing loads (kip):", bearing_loads(0))

Strength I unit reactions (kip): {'Abutment 1': 243, 'Pier 2': 883, 'Pier 3': 883, 'Abutment 2': 243}
Pier bearing loads (kip):     [193.1, 248.3, 248.3, 193.1]
Abutment bearing loads (kip): [53.1, 68.3, 68.3, 53.1]


## 2. Cap designs from the STM topology optimizer

Every cap in the gallery is a **real executed design**:
`optimize_pier_cap` sweeps candidate depths, runs the structured-quad
topology optimization, extracts the strut-and-tie model, checks strut
angles and nodal capacities, and prices the result. The hammerhead solves
as a double cantilever off its single stem — the governing tie lands in
the **top** chord, and `hammerhead_geometry` reads that height to taper
the soffit. (~25 s for the three sweeps.)

In [3]:
from civilpy.structural.stm_topology.design import optimize_pier_cap

hh_cap = optimize_pier_cap(bearing_loads(1), load_xs=bearing_y,
                           column_xs=[13.5], f_c=4.0, thickness=5.0,
                           depth_bounds=(5.0, 9.0), n_depths=5,
                           column_bearing=6.0)     # 72 in stem
print(hh_cap.summary())

pb_piles = (2.0, 13.5, 25.0)
pb_cap = optimize_pier_cap(bearing_loads(2), load_xs=bearing_y,
                           column_xs=list(pb_piles), f_c=4.0, thickness=3.5,
                           depth_bounds=(4.0, 7.5), n_depths=4, nelx=84,
                           column_bearing=1.5)     # HP12 pile head
print()
print(pb_cap.summary())

ab_piles = (2.0, 13.5, 25.0)
ab_cap = optimize_pier_cap(bearing_loads(0), load_xs=bearing_y,
                           column_xs=list(ab_piles), f_c=4.0, thickness=3.5,
                           depth_bounds=(3.5, 6.5), n_depths=4, nelx=84,
                           column_bearing=1.0)     # HP10 pile head
print()
print(ab_cap.summary())

Pier-cap depth optimization (span 32 ft, thickness 5 ft):
  optimum depth = 7.0 ft  ->  $36,130  (concrete $35,259 + 725 lb steel)
  governing: strut angle 27.4 deg >= 25 (a/d limit); nodal capacity/demand 5.28; governing tie 532 kip
  feasible depths: 7.0-9.0 ft



Pier-cap depth optimization (span 32 ft, thickness 4 ft):
  optimum depth = 6.3 ft  ->  $22,611  (concrete $22,331 + 233 lb steel)
  governing: strut angle 54.6 deg >= 25 (a/d limit); nodal capacity/demand 3.62; governing tie 237 kip
  feasible depths: 6.3-7.5 ft



Pier-cap depth optimization (span 32 ft, thickness 4 ft):
  optimum depth = 4.5 ft  ->  $16,140  (concrete $15,867 + 227 lb steel)
  governing: strut angle 45.0 deg >= 25 (a/d limit); nodal capacity/demand 9.36; governing tie 113 kip
  feasible depths: 4.5-6.5 ft


## 3. One of each type on one bridge

`assemble_substructure` maps each support-line index to its typed spec:

| unit | index | type | spec |
|---|---|---|---|
| Abutment 1 | 0 | **semi-integral** | `SemiIntegralAbutmentSpec` — seat abutment, backwall replaced by a superstructure-borne end diaphragm |
| Pier 2 | 1 | **hammerhead** | `HammerheadSpec` — tapered cap on a single 6×5 ft stem, spread footing |
| Pier 3 | 2 | **capped-pile bent** | `PileBentSpec` — cap directly on driven HP12X53 piles, no columns |
| Abutment 2 | 3 | **integral** | `IntegralAbutmentSpec` — full-height end diaphragm cast around the girder ends on one HP10X42 pile row, **no bearings** |

In [4]:
from civilpy.structural.abutment import RetainingWall
from civilpy.structural.aashto.lrfd.columns import RebarLayer
from civilpy.structural.pier import PierColumn
from civilpy.structural.substructure_layout import (
    AbutmentSpec, FootingSpec, HammerheadSpec, IntegralAbutmentSpec,
    PileBentSpec, SemiIntegralAbutmentSpec, assemble_substructure)

stem = PierColumn(height=22.0 * 12.0, b=72.0, h=60.0, f_c=4.0,
                  layers=[RebarLayer(area=10.0, depth=6.0),
                          RebarLayer(area=10.0, depth=54.0)])
wall = RetainingWall(stem_height=14.0, stem_thickness=1.5, toe_length=4.0,
                     heel_length=8.0, footing_thickness=3.0,
                     backfill_gamma=120.0, backfill_phi=32.0)

sub = assemble_substructure(layout, {
    0: SemiIntegralAbutmentSpec(
        cap_design=ab_cap,
        spec=AbutmentSpec(pile_xs_ft=ab_piles, pile_shape="HP10X42",
                          pile_length_ft=40.0,
                          wingwall=wall, wingwall_length_ft=12.0),
        diaphragm_thickness_in=30.0),
    1: HammerheadSpec(cap_design=hh_cap, column=stem, tip_depth_ft=3.0,
                      footing=FootingSpec(16.0, 16.0, 4.0)),
    2: PileBentSpec(cap_design=pb_cap, pile_xs_ft=pb_piles,
                    pile_shape="HP12X53", pile_length_ft=40.0),
    3: IntegralAbutmentSpec(pile_xs_ft=(2.0, 10.0, 17.0, 25.0),
                            pile_shape="HP10X42", pile_length_ft=40.0,
                            wingwall=wall, wingwall_length_ft=12.0),
})

for a in sub.abutments:
    print(f"{a.unit.name:<12} -> {a.kind} abutment, "
          f"{len(a.piles)} piles, {len(a.wingwalls)} wingwall panels")
for p in sub.piers:
    kind = "hammerhead" if (len(p.columns) == 1 and not p.piles) else \
           ("capped-pile bent" if p.piles else "column bent")
    print(f"{p.unit.name:<12} -> {kind}, cap {p.cap.depth_ft:.1f} ft deep")

Abutment 1   -> semi-integral abutment, 3 piles, 4 wingwall panels
Abutment 2   -> integral abutment, 4 piles, 4 wingwall panels
Pier 2       -> hammerhead, cap 7.0 ft deep
Pier 3       -> capped-pile bent, cap 6.3 ft deep


## 4. Emit the full BrIM model

`girder_bridge_emit` builds the tagged superstructure (girders with true
k-fillets, crowned deck, haunches, studs, deck mats, SBR-1-20 parapets
with their SCD cage, bearings) — `integral_supports=(3,)` skips the
bearing stack at Abutment 2, because there the girder ends are cast into
the diaphragm. `add_substructure` appends the placed units with their
rebar cages read from the executed STM designs. Every object carries its
pay item, so the estimate is a walk over the model.

In [5]:
from civilpy.structural.rhino_bim import (
    add_substructure, emit_to_3dm, emit_to_json, girder_bridge_emit,
    pay_item_quantities, read_bim_quantities)

emit = girder_bridge_emit(inp, integral_supports=(3,))
full = add_substructure(emit, sub)
print(f"{len(full.objects):,} tagged objects\n")

print(f"{'item':<12}{'qty':>12} {'unit':<5} {'description'}")
for item, rec in pay_item_quantities(full).items():
    print(f"{item:<12}{rec['qty']:>12,.1f} {rec['unit']:<5} {rec['desc']}")

4,468 tagged objects

item                 qty unit  description
507E10000          400.0 ft    Steel piles HP, furnished and driven [CONFIRM]
509E00200       61,671.3 lb    Epoxy coated reinforcing steel [CONFIRM]
509E00300            0.0 lb    GFRP deformed bars [CONFIRM]
511E12100          208.8 cy    Class QC2 concrete, superstructure (deck) [CONFIRM]
511E40000          249.3 cy    Class QC1 concrete, substructure [CONFIRM]
512E10000           60.5 cy    Concrete, parapet/railing [CONFIRM]
513E10220      122,250.9 lb    Structural steel members, Level 1
513E20000        1,200.0 ea    Shear connectors (welded studs)
516E10000           12.0 ea    Elastomeric bearing [CONFIRM]


## 5. Bake the Rhino file — no Rhino, no MCP, no network

`emit_to_3dm` writes the model with standalone `rhino3dm`: prisms become
capped extrusion breps, studs cylinder breps, rebar polyline curves, on
the same colored layer taxonomy the live driver uses, every object
stamped with its full user-text tags. **Open
`substructure_gallery.3dm` in Rhino and the gallery is there.**

The JSON payload is written alongside it for the live path: in Rhino 8's
ScriptEditor, open `Rhino Components/draw_bim_emit.py`, set
`EMIT_JSON_PATH` to the payload, and run — same model, drawn into the
open document with the bridge viewport setup.

In [6]:
counts = emit_to_3dm(full, "substructure_gallery.3dm")
print(f"substructure_gallery.3dm: {sum(counts.values()):,} objects\n")
for layer in sorted(counts):
    print(f"  {layer:<28} {counts[layer]:>5}")

with open("substructure_gallery_emit.json", "w") as f:
    f.write(emit_to_json(full))
print("\nwrote substructure_gallery_emit.json for draw_bim_emit.py")

substructure_gallery.3dm: 4,468 objects

  Deck::Bridge Deck                2
  Deck::Rebar                   2569
  Deck::Traffic Barriers           2
  Substructure::Beam Seats        12
  Substructure::Caps               3
  Substructure::Columns            1
  Substructure::Footings           1
  Substructure::Piles             10
  Substructure::Rebar            606
  Substructure::Wingwalls          8
  Superstructure::Bearings        28
  Superstructure::Diaphragms       2
  Superstructure::Girders          8
  Superstructure::Haunches         4
  Superstructure::Load Plates     12
  Superstructure::Shear Studs   1200

wrote substructure_gallery_emit.json for draw_bim_emit.py


## 6. The saved file is the record

`read_bim_quantities` regenerates the estimate **from the `.3dm` alone**
— no civilpy state, no session that drew it. That round trip is the
source-of-truth contract: the Rhino document, not the notebook run,
carries the engineering record.

In [7]:
q_model = pay_item_quantities(full)
q_file = read_bim_quantities("substructure_gallery.3dm")

assert set(q_model) == set(q_file)
for item in q_model:
    assert abs(q_model[item]["qty"] - q_file[item]["qty"]) < 1e-6, item
print("pay-item rollup from the saved .3dm matches the emit exactly:")
for item, rec in q_file.items():
    print(f"  {item}  {rec['qty']:>12,.1f} {rec['unit']}")

pay-item rollup from the saved .3dm matches the emit exactly:
  507E10000         400.0 ft
  509E00200      61,671.3 lb
  509E00300           0.0 lb
  511E12100         208.8 cy
  511E40000         249.3 cy
  512E10000          60.5 cy
  513E10220     122,250.9 lb
  513E20000       1,200.0 ea
  516E10000          12.0 ea


---

## Where this leaves us

- **Every substructure type in one document**, each placed from a real
  executed design: the STM optimizer sized the caps, the tie schedule
  drove the rebar cages, and the integral end killed its bearing stack.
- **Two Rhino backends, zero infrastructure**: `emit_to_3dm` bakes the
  file headlessly (this notebook, any machine with `pip install
  rhino3dm`); `draw_bim_emit.py` paints a live Rhino 8 document from the
  same JSON. The MCP server the prototypes used was just a third
  transport for the same payload — nothing about the models needs it.
- The `gdr.*` tags on girder centerlines and bearing points survive in
  the baked file, so `rhino_gdr` / the MIDAS pipeline can read the
  document back into analysis — the same round trip the *Rhino to MIDAS
  Pipeline Verification* notebook exercises.